In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### create a table saving the info of the new stations want to add

### Need to be careful about the network id in the new run

In [5]:
import pandas as pd

# Manually define station metadata extracted from the paragraph
data = [
    {
        "Native_ID": "452",
        "Station_ID": None,
        "Unique_Name": "Fraser Lake Endako Mines",
        "Network_Name": "BC-AIR",
        "Network_ID": 9,
        "Latitude": 54.034444,
        "Longitude": -125.095833,
        "Elevation_m": 1005
    },
    {
        "Native_ID": "543",
        "Station_ID": None,
        "Unique_Name": "Skookumchuk Farstad Way",
        "Network_Name": "BC-AIR",
        "Network_ID": 9,
        "Latitude": 49.905404,
        "Longitude": -115.755613,
        "Elevation_m": 746
    },
    {
        "Native_ID": "T035",
        "Station_ID": None,
        "Unique_Name": "Horseshoe Bay",
        "Network_Name": "MVRD-AIR",
        "Network_ID": 137,
        "Latitude": 49.3686,
        "Longitude": -123.2766,
        "Elevation_m": 65
    }
]

# Create DataFrame
stations_df = pd.DataFrame(data)

stations_df

,Native_ID,Station_ID,Unique_Name,Network_Name,Network_ID,Latitude,Longitude,Elevation_m
0,452,None,Fraser Lake Endako Mines,BC-AIR,9,54.034444,-125.095833,1005
1,543,None,Skookumchuk Farstad Way,BC-AIR,9,49.905404,-115.755613,746
2,T035,None,Horseshoe Bay,MVRD-AIR,137,49.368600,-123.276600,65


In [6]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in stations_df.iterrows():
        native_id = row['Native_ID']
        network_id = row['Network_ID']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(stations_df)}] "
            f"old_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
stations_df['old_station_status'] = exists_list

[  1/3] old_native_id=452             | → EXISTS
[  2/3] old_native_id=543             | → NOT EXISTS
[  3/3] old_native_id=T035            | → EXISTS
